### install the dependencies:

In [1]:
!uv init --no-workspace

error: Project is already initialized in `/workspaces/llm_zoomcamp`
       (`pyproject.toml` file exists)


In [2]:
!uv add onnxruntime tokenizers numpy tqdm minsearch gitsource

Resolved 174 packages in 1ms
Checked 170 packages in 43ms


In [3]:
!uv add --dev huggingface-hub jupyter

Resolved 174 packages in 0.80ms
Checked 170 packages in 2ms


In [4]:
PREFIX="https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/02-vector-search/embed"
!wget {PREFIX}/download.py
!wget {PREFIX}/embedder.py

--2026-06-29 18:30:30--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/02-vector-search/embed/download.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.108.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1376 (1.3K) [text/plain]
Saving to: ‘download.py.1’

download.py.1       100%[===================>]   1.34K  --.-KB/s    in 0s      

2026-06-29 18:30:30 (81.5 MB/s) - ‘download.py.1’ saved [1376/1376]

--2026-06-29 18:30:30--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/02-vector-search/embed/embedder.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.111.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 

In [5]:
!uv run python download.py

tokenizer.json: 100%|███████████████████████| 712k/712k [00:00<00:00, 85.8MB/s]
  saved models/Xenova/all-MiniLM-L6-v2/tokenizer.json
onnx/model.onnx: 100%|████████████████████| 90.4M/90.4M [00:01<00:00, 48.1MB/s]
  saved models/Xenova/all-MiniLM-L6-v2/model.onnx


### Q1. Embedding a query

In [6]:
q = "How does approximate nearest neighbor search work?"

In [7]:
from embedder import Embedder

embedder = Embedder()

q = "How does approximate nearest neighbor search work?"

q_emb = embedder.encode(q)

print(type(q_emb))
print(q_emb.shape)
print(q_emb[:10])

2026-06-29 18:30:35.155643643 [W:onnxruntime:Default, device_discovery.cc:133 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


<class 'numpy.ndarray'>
(384,)
[-0.02058203 -0.01404588  0.03029941 -0.05403784  0.07187811 -0.02795375
 -0.05030938 -0.01272173  0.04082079 -0.02600374]


### Q2. Cosine similarity

In [8]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [10]:
import numpy as np

In [12]:
print(documents[0].keys())

dict_keys(['content', 'filename'])


In [13]:
target_doc = next(
    doc for doc in documents
    if doc["filename"] == "02-vector-search/lessons/07-sqlitesearch-vector.md"
)


In [17]:
target_doc

{'content': '# Vector Search with sqlitesearch\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=csxKescwJYM&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous section we used minsearch for vector search.\n\nIt works, but it has three problems:\n\n- It rebuilds the index on every startup\n- It keeps everything in memory\n- It searches by brute force\n\n\nWith text search we never felt these. Indexing was fast because we\ndidn\'t embed anything. With vector search, indexing runs a neural\nnetwork over every document, so it takes a minute on our dataset.\nKeeping everything in memory is fine here, but a larger dataset would\nneed too much space.\n\nThe third problem is brute-force search. For every query we compare the\nquery vector against every single document. With 1,000 documents this is\nfine, probably even faster than anything smarter. But as the dataset\ngrows past 10,000 or so, it gets slow, and we\'ll want an approximate\nmethod instead.\n\nWhat we\'ve done 

In [19]:
doc_text = target_doc["content"]
doc_emb = embedder.encode(doc_text)

np.dot(q_emb, doc_emb)

np.float64(0.36107026789538205)

### Q3. Chunking and search by hand

In [20]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)

In [23]:
texts = [chunk["content"] for chunk in chunks]
X = embedder.encode_batch(texts)  

In [24]:
texts = [c["content"] for c in chunks]
X = embedder.encode_batch(texts)

q = "How does approximate nearest neighbor search work?"
v = embedder.encode(q)

scores = X.dot(v)

top_idx = np.argmax(scores)

best_chunk = chunks[top_idx]
print(best_chunk["filename"])
print(scores[top_idx])

02-vector-search/lessons/07-sqlitesearch-vector.md
0.6489016436447387


### Q4. Vector search with minsearch

In [26]:
from minsearch import VectorSearch

In [27]:
texts = [c["content"] for c in chunks]
vectors = embedder.encode_batch(texts)

In [32]:
index = VectorSearch(keyword_fields=["course"])

In [33]:
texts = [c["content"] for c in chunks]
vectors = embedder.encode_batch(texts)

index.fit(vectors, chunks)

In [35]:
q = "What metric do we use to evaluate a search engine?"
q_vec = embedder.encode(q)

results = index.search(q_vec)
print(results[0]["filename"])

04-evaluation/lessons/05-search-metrics.md


### Q5. Text search vs vector search

In [60]:
from minsearch import Index

text_index = Index(text_fields=["content"])
text_index.fit(chunks)

In [61]:
texts = [c["content"] for c in chunks]
vectors = embedder.encode_batch(texts)

In [62]:
vector_index = VectorSearch()
texts = [c["content"] for c in chunks]
vectors = embedder.encode_batch(texts)
vector_index.fit(vectors, chunks)

In [63]:
query = "How do I store vectors in PostgreSQL?"
q_vec = embedder.encode(query)

In [76]:
vector_results = vector_index.search(q_vec)
top_vector = [r["filename"] for r in vector_results[:5]]
top_vector

['02-vector-search/lessons/08-pgvector.md',
 '02-vector-search/lessons/08-pgvector.md',
 '03-orchestration/lessons/05-rag.md',
 '02-vector-search/lessons/08-pgvector.md',
 '02-vector-search/lessons/08-pgvector.md']

In [ ]:
text_results = text_index.search(query)
top_text = [r["filename"] for r in text_results[:5]]
top_text
##02-vector-search/lessons/08-pgvector.md shows up in the vector results but not in the text results

['02-vector-search/lessons/02-embeddings.md',
 '03-orchestration/lessons/05-rag.md',
 '02-vector-search/lessons/01-intro.md',
 '03-orchestration/lessons/05-rag.md',
 '02-vector-search/lessons/01-intro.md']

#### Q6. Hybrid search

In [79]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [92]:
query = "How do I give the model access to tools?"
text_results = text_index.search(query)

q_vec = embedder.encode(query)
vector_results = vector_index.search(q_vec)

In [93]:
text_results = text_index.search(query)
vector_results = vector_index.search(q_vec)

results = rrf([vector_results, text_results])

In [94]:
results[0]["filename"]

'01-agentic-rag/lessons/13-function-calling.md'